# Experiment 1.3b-ii-A: Off-policy Hessian eigendirection alignment of SGD vs Muon updates

**Pair counterpart:** `run_experiment.py` in the same directory is the source of truth for the experiment core. This notebook imports that script, runs its `run_experiment(...)` function, and analyzes the returned structured results rather than re-implementing the finite-difference Hessian probe.

## Honest scope of this first-pass artifact

- **Toy setting:** a `2 -> 2 -> 1` deep linear network with only 6 parameters.
- **Trajectory policy:** the model is trained with **SGD only**.
- **Muon role in this pair:** a **hypothetical off-policy update** evaluated at the same SGD checkpoints.
- **Primary flatness-related diagnostic:** squared projection mass on the two Hessian eigendirections with the **smallest absolute eigenvalues** (`2 smallest |λ|`).
- **Secondary continuity diagnostic:** squared projection mass on the two **most negative signed-eigenvalue** directions. This legacy metric is kept for comparison only and is **not** treated as a literal gauge-space measurement.
- **Interpretive limit:** this notebook does **not** construct an explicit gauge tangent basis, and therefore does **not** present a completed mechanistic explanation of the broader Muon/gauge-fixing story.


## Counterpart identity and what this pair actually computes

This pair is intentionally narrow. At selected checkpoints along an SGD training trajectory, the script:

1. computes the **full finite-difference Hessian** of the 6-parameter model,
2. eigendecomposes that Hessian,
3. compares the **SGD update direction** to the **Muon-style orthogonalized update direction** that would be taken **at the same point**, and
4. measures how much of each update falls on selected Hessian eigendirections.

So the object of study is **directional alignment at shared parameter points**, not a direct SGD-vs-Muon training race.


In [ ]:
from pathlib import Path
import importlib.util
import time

import numpy as np
import matplotlib.pyplot as plt

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from IPython.display import display, Markdown
except ImportError:
    def display(obj):
        print(obj)
    def Markdown(text):
        return text

plt.style.use('seaborn-v0_8-whitegrid')

EXPERIMENT_REL = Path(
    'experiments/Tier_1_Core_Mechanism_Tests/'
    'Exp_1.3_Singular_Value_Spectrum_Evolution/'
    '1.3b_Spectrum_Gradient_vs_Weight_Spectrum_Correlation/'
    '1.3b-ii_Hessian_Alignment_Does_Muon_Update_Avoid_Flat_Directions/'
    '1.3b-ii-A_Hessian_Top-5_Eigenvector_Alignment_With_Update'
)


def locate_experiment_dir():
    cwd = Path.cwd().resolve()
    if (cwd / 'run_experiment.py').exists():
        return cwd
    for base in [cwd, *cwd.parents]:
        candidate = base / EXPERIMENT_REL / 'run_experiment.py'
        if candidate.exists():
            return candidate.parent
    raise FileNotFoundError('Could not locate the experiment directory from the current working directory.')


def make_df(rows, index=None):
    if pd is None:
        return rows
    df = pd.DataFrame(rows)
    if index is not None and index in df.columns:
        df = df.set_index(index)
    return df


EXPERIMENT_DIR = locate_experiment_dir()
SCRIPT_PATH = EXPERIMENT_DIR / 'run_experiment.py'
spec = importlib.util.spec_from_file_location('exp_1_3b_ii_A_run_experiment', SCRIPT_PATH)
exp = importlib.util.module_from_spec(spec)
spec.loader.exec_module(exp)

print(f'Experiment directory: {EXPERIMENT_DIR}')
print(f'Script path:          {SCRIPT_PATH}')


## Reproducibility, runtime, and configuration

The next cell runs the script code path directly. This is the notebook's main reproducibility cell: it records the seed, configuration, output paths, and measured runtime, while keeping the script as the sole source of truth for the experiment core.


In [ ]:
seed = exp.DEFAULT_SEED
config = exp.get_default_config()
save_script_plots = False  # keep notebook execution side-effect-light; inline figures below are primary

notebook_t0 = time.time()
results = exp.run_experiment(
    config=config,
    seed=seed,
    verbose=False,
    save_plots=save_script_plots,
    output_dir=EXPERIMENT_DIR,
)
notebook_wall_clock = time.time() - notebook_t0

repro_rows = [
    {'field': 'experiment_id', 'value': results['experiment_id']},
    {'field': 'title', 'value': results['title']},
    {'field': 'seed', 'value': results['seed']},
    {'field': 'trajectory_policy', 'value': results['trajectory_policy']},
    {'field': 'muon_evaluation_mode', 'value': results['muon_evaluation_mode']},
    {'field': 'primary_metric', 'value': results['verdict']['primary_metric']},
    {'field': 'secondary_metric', 'value': results['verdict']['secondary_metric']},
    {'field': 'save_script_plots', 'value': save_script_plots},
    {'field': 'script_reported_runtime_seconds', 'value': round(results['runtime_seconds'], 4)},
    {'field': 'notebook_wall_clock_seconds', 'value': round(notebook_wall_clock, 4)},
    {'field': 'output_plot_paths', 'value': results['plot_paths']},
]

display(make_df(repro_rows, index='field'))


In [ ]:
config_rows = [
    {'setting': 'n_samples', 'value': results['config']['n_samples']},
    {'setting': 'network', 'value': f"{results['config']['input_dim']} -> {results['config']['hidden_dim']} -> {results['config']['output_dim']}"},
    {'setting': 'n_params', 'value': results['config']['n_params']},
    {'setting': 'epsilon', 'value': results['config']['epsilon']},
    {'setting': 'lr_sgd', 'value': results['config']['lr_sgd']},
    {'setting': 'n_training_steps', 'value': results['config']['n_training_steps']},
    {'setting': 'measure_steps', 'value': results['config']['measure_steps']},
    {'setting': 'top_k', 'value': results['config']['top_k']},
    {'setting': 'low_curvature_k', 'value': results['config']['low_curvature_k']},
    {'setting': 'legacy_bottom_signed_k', 'value': results['config']['legacy_bottom_signed_k']},
]

data_rows = [
    {'quantity': 'X shape', 'value': results['data_summary']['x_shape']},
    {'quantity': 'Y shape', 'value': results['data_summary']['y_shape']},
    {'quantity': 'X mean', 'value': round(results['data_summary']['x_mean'], 6)},
    {'quantity': 'X std', 'value': round(results['data_summary']['x_std'], 6)},
    {'quantity': 'Y mean', 'value': round(results['data_summary']['y_mean'], 6)},
    {'quantity': 'Y std', 'value': round(results['data_summary']['y_std'], 6)},
    {'quantity': 'eig(X^T X)', 'value': np.round(results['data_summary']['xtx_eigenvalues'], 6)},
]

print(results['scope_note'])
display(make_df(config_rows, index='setting'))
display(make_df(data_rows, index='quantity'))


## What is measured at each checkpoint

At each measurement step along the SGD trajectory, the script returns:

- the Hessian eigenvalues (sorted by **signed eigenvalue**, descending),
- the normalized absolute projections of the SGD update onto each eigenvector,
- the normalized absolute projections of the hypothetical Muon update onto each eigenvector,
- summary masses on the **top-curvature** directions, the **low-curvature proxy** directions (`2 smallest |λ|`), and the **legacy signed-bottom** directions.

The table below is the main per-step record used for interpretation in this notebook.


In [ ]:
per_step_rows = []
for r in results['per_step']:
    per_step_rows.append({
        'step': r['step'],
        'loss': round(r['loss'], 6),
        'top_slots': [idx + 1 for idx in r['direction_indices']['top']],
        'low_curvature_slots': [idx + 1 for idx in r['direction_indices']['low_curvature']],
        'legacy_bottom_signed_slots': [idx + 1 for idx in r['direction_indices']['legacy_bottom_signed']],
        'top_mass_sgd': round(r['top_mass_sgd'], 6),
        'top_mass_muon': round(r['top_mass_muon'], 6),
        'low_curvature_mass_sgd': round(r['low_curvature_mass_sgd'], 6),
        'low_curvature_mass_muon': round(r['low_curvature_mass_muon'], 6),
        'legacy_bottom_signed_mass_sgd': round(r['legacy_bottom_signed_mass_sgd'], 6),
        'legacy_bottom_signed_mass_muon': round(r['legacy_bottom_signed_mass_muon'], 6),
        'update_cosine': round(r['update_cosine'], 6),
        'update_angle_deg': round(r['update_angle_deg'], 4),
    })

per_step_df = make_df(per_step_rows, index='step')
display(per_step_df)


## Notebook-native figures: primary and secondary summaries

The first figure emphasizes the **primary** pair-level comparison requested in this pass: how much update mass SGD and Muon place on the top-curvature directions and on the low-curvature proxy directions across the measured SGD checkpoints.

The second panel keeps the legacy signed-bottom diagnostic visible, but only as a secondary continuity check.


In [ ]:
steps = np.array([r['step'] for r in results['per_step']])
low_sgd = np.array([r['low_curvature_mass_sgd'] for r in results['per_step']])
low_muon = np.array([r['low_curvature_mass_muon'] for r in results['per_step']])
top_sgd = np.array([r['top_mass_sgd'] for r in results['per_step']])
top_muon = np.array([r['top_mass_muon'] for r in results['per_step']])
legacy_sgd = np.array([r['legacy_bottom_signed_mass_sgd'] for r in results['per_step']])
legacy_muon = np.array([r['legacy_bottom_signed_mass_muon'] for r in results['per_step']])

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

axes[0].plot(steps, low_sgd, 'o-', color='steelblue', label='SGD')
axes[0].plot(steps, low_muon, 'o-', color='firebrick', label='Muon')
axes[0].set_title('Primary low-curvature proxy mass\n(2 smallest |λ| directions)')
axes[0].set_xlabel('SGD training step')
axes[0].set_ylabel('Sum of squared projections')
axes[0].legend()

axes[1].plot(steps, top_sgd, 'o-', color='steelblue', label='SGD')
axes[1].plot(steps, top_muon, 'o-', color='firebrick', label='Muon')
axes[1].set_title('Top-curvature mass\n(2 largest signed λ directions)')
axes[1].set_xlabel('SGD training step')
axes[1].set_ylabel('Sum of squared projections')
axes[1].legend()

axes[2].plot(steps, legacy_sgd, 'o-', color='steelblue', label='SGD')
axes[2].plot(steps, legacy_muon, 'o-', color='firebrick', label='Muon')
axes[2].set_title('Secondary legacy diagnostic\n(2 most negative signed λ directions)')
axes[2].set_xlabel('SGD training step')
axes[2].set_ylabel('Sum of squared projections')
axes[2].legend()

fig.suptitle(
    'Per-step update mass on selected Hessian eigendirections\n'
    'Muon is evaluated off-policy at SGD checkpoints',
    fontsize=13,
    fontweight='bold',
)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()


In [ ]:
avg_proj_sgd = results['aggregate']['avg_proj_sgd']
avg_proj_muon = results['aggregate']['avg_proj_muon']
avg_evals = np.mean([r['eigenvalues'] for r in results['per_step']], axis=0)

fig, ax = plt.subplots(figsize=(10, 4.8))
x = np.arange(len(avg_proj_sgd))
width = 0.35
ax.bar(x - width / 2, avg_proj_sgd, width, color='steelblue', alpha=0.85, label='SGD')
ax.bar(x + width / 2, avg_proj_muon, width, color='firebrick', alpha=0.85, label='Muon')
ax.set_xticks(x)
ax.set_xticklabels([f"v{i + 1}\n{lam:.3f}" for i, lam in enumerate(avg_evals)])
ax.set_xlabel('Eigenvector rank (sorted by signed λ, descending)')
ax.set_ylabel('Average |projection| / ||update||')
ax.set_title('Average alignment by Hessian eigenvector rank')
ax.legend()
plt.tight_layout()
plt.show()

print('Script-generated plot files (empty unless save_script_plots=True):')
for name, plot_path in results['plot_paths'].items():
    print(f'  {name}: {plot_path}')
if not results['plot_paths']:
    print('  <none written by notebook run>')


## Aggregate summary tables

The table below is the key pair-level comparison. The primary line is the low-curvature proxy mass. The top-curvature line is useful context, and the legacy signed-bottom line is kept only for continuity with earlier wording.


In [ ]:
agg = results['aggregate']
summary_rows = [
    {
        'metric': 'Average top-curvature mass (2 largest λ)',
        'SGD': agg['avg_top_mass_sgd'],
        'Muon': agg['avg_top_mass_muon'],
        'Muon/SGD': agg['top_mass_ratio'],
    },
    {
        'metric': 'Average low-curvature proxy mass (2 smallest |λ|) [primary]',
        'SGD': agg['avg_low_curvature_mass_sgd'],
        'Muon': agg['avg_low_curvature_mass_muon'],
        'Muon/SGD': agg['low_curvature_mass_ratio'],
    },
    {
        'metric': 'Average legacy bottom-signed mass (2 most negative λ) [secondary]',
        'SGD': agg['avg_legacy_bottom_signed_mass_sgd'],
        'Muon': agg['avg_legacy_bottom_signed_mass_muon'],
        'Muon/SGD': agg['legacy_bottom_signed_mass_ratio'],
    },
]

verdict_rows = [
    {'component': 'verdict', 'value': results['verdict']['label']},
    {'component': 'primary interpretation', 'value': results['verdict']['interpretation']},
    {'component': 'Muon lower low-curvature mass steps', 'value': f"{agg['muon_less_low_curvature_steps']}/{agg['n_measurements']}"},
    {'component': 'Muon higher top-curvature mass steps', 'value': f"{agg['muon_more_top_steps']}/{agg['n_measurements']}"},
    {'component': 'Muon lower legacy bottom-signed mass steps', 'value': f"{agg['muon_less_legacy_bottom_signed_steps']}/{agg['n_measurements']}"},
    {'component': 'mean cosine(SGD, Muon)', 'value': round(agg['avg_update_cosine'], 6)},
    {'component': 'mean angle(SGD, Muon) [deg]', 'value': round(agg['avg_update_angle_deg'], 4)},
]

display(make_df(summary_rows, index='metric'))
display(make_df(verdict_rows, index='component'))


## Interpretation

A serious first-pass reading of this pair should keep three distinctions explicit:

1. **Primary evidence source:** the low-curvature proxy based on the smallest-`|λ|` directions.
2. **Secondary continuity source:** the old signed-bottom diagnostic, which may behave differently because negative-curvature directions are not the same thing as low-curvature or literal gauge directions.
3. **Off-policy status:** Muon is never allowed to generate its own trajectory here; we only compare what Muon would do locally at points selected by SGD.


In [ ]:
agg = results['aggregate']
verdict = results['verdict']['label']

conclusion_lines = [
    '### Calibrated conclusion',
    '',
    f"- **Outcome under the current primary metric:** **{verdict}**.",
    f"- **Primary metric used in this notebook/script pass:** {results['verdict']['primary_metric']}.",
    f"- **Average low-curvature proxy mass:** SGD = {agg['avg_low_curvature_mass_sgd']:.6f}, Muon = {agg['avg_low_curvature_mass_muon']:.6f}, Muon/SGD = {agg['low_curvature_mass_ratio']:.3f}.",
    f"- **Average top-curvature mass:** SGD = {agg['avg_top_mass_sgd']:.6f}, Muon = {agg['avg_top_mass_muon']:.6f}, Muon/SGD = {agg['top_mass_ratio']:.3f}.",
    f"- **Legacy signed-bottom diagnostic:** SGD = {agg['avg_legacy_bottom_signed_mass_sgd']:.6f}, Muon = {agg['avg_legacy_bottom_signed_mass_muon']:.6f}, Muon/SGD = {agg['legacy_bottom_signed_mass_ratio']:.3f}.",
    f"- **Step-count summary:** Muon has lower low-curvature proxy mass on {agg['muon_less_low_curvature_steps']}/{agg['n_measurements']} measured checkpoints, but higher top-curvature mass on only {agg['muon_more_top_steps']}/{agg['n_measurements']} checkpoints.",
    '',
    'Therefore, this first-pass pair does **not** justify a strong claim that Muon cleanly "avoids gauge directions" in this toy problem. What it does provide is a compact, reproducible, off-policy geometric probe showing how the Muon-style orthogonalized update differs from the SGD update when both are evaluated at the same SGD trajectory points.',
]
conclusion = '\n'.join(conclusion_lines)

try:
    display(Markdown(conclusion))
except Exception:
    print(conclusion)


## Caveats and best next upgrade

The main unresolved issue is still conceptual rather than computational: **low-curvature Hessian directions are only a proxy for gauge-like directions**. A stronger next step would construct the explicit gauge tangent subspace for the hidden-layer transformation symmetry and measure update mass in that basis directly. Until then, the current notebook/script pair should be read as a careful **toy Hessian-eigendirection study**, not as a definitive mechanistic explanation.
